# 3.4 Dynamic Tools — Role-Based Tool Access

## What you will learn in this notebook

- Why **tool access control** matters for multi-user applications
- How **`@wrap_model_call`** restricts which tools each user can see
- How to implement **role-based access control (RBAC)** for agent tools
- How external users are silently blocked from internal tools

---

### The problem: all users sharing the same agent

In production, one agent serves many users. But not all users should have access to all tools:

| User type | Should see | Should NOT see |
|---|---|---|
| **External customer** | Web search | Internal DB queries |
| **Internal analyst** | Web search + SQL | — |
| **Admin** | Everything | — |

If you give all tools to all users, an external user can ask the agent to run SQL on your internal database. That's a security problem.

### How dynamic tools solve it

Instead of creating separate agents per role (expensive, unmaintainable), we use `@wrap_model_call` to **filter the tool list per request** based on the user's role in runtime context:

```
Same agent object
    ↓
request.runtime.context.user_role = "external"
    ↓
middleware: tools = [web_search]          ← SQL removed
    ↓
LLM sees only web_search in tool list
    ↓
Agent cannot call sql_query even if asked
```

In [3]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# ============================================================
# CELL 2: Define Two Tools With Different Risk Profiles
# ============================================================
# web_search (LOW sensitivity):
#   → Searches the public internet — safe for all users
#   → No access to internal data
#
# sql_query (HIGH sensitivity):
#   → Queries the internal Chinook database
#   → Contains proprietary business data
#   → Should only be available to internal users with appropriate access
#   → Wrapped in try/except so SQL errors don't crash the agent

from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()
db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
# ============================================================
# CELL 3: Define the User Role Context
# ============================================================
# UserRole is the context schema injected per request.
# user_role defaults to "external" — the most restrictive tier.
# This is a safe default: if no role is provided, the user
# gets the minimum set of tools, not the maximum.
#
# In production, this would be set from your auth middleware:
#   decoded_token = verify_jwt(request.headers["Authorization"])
#   context = UserRole(user_role=decoded_token["role"])
#   agent.invoke(messages, context=context)

from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"   # Default: most restrictive — safe fallback

In [6]:
# ============================================================
# CELL 4: Define the Dynamic Tool Selection Middleware
# ============================================================
# @wrap_model_call intercepts each LLM call and adjusts the tool list.
#
# user_role = request.runtime.context.user_role
#   → Reads the role from runtime context (injected at invoke() time)
#
# if user_role == "internal":
#   pass  → Don't modify the request — internal users get ALL tools
#           (the full list was already set in create_agent(tools=[...]))
#
# else:
#   tools = [web_search]  → External users get ONLY web_search
#   request = request.override(tools=tools)  → Replace the tool list
#
# Why `pass` for internal users rather than explicitly setting all tools?
#   Because the full tool list is already in the request from create_agent().
#   We only need to RESTRICT for external users — no action for internal.
#   This pattern scales: add more roles without changing the base agent.
#
# Security note: the override happens BEFORE the LLM receives the request.
# The LLM never sees sql_query in its tool list for external users.
# Even if a user crafts a prompt like "use the sql_query tool to...",
# the LLM will respond that it has no such tool.

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass   # Internal users keep all tools — no modification needed
    else:
        # External users: restrict to web_search only
        tools = [web_search]
        request = request.override(tools=tools)   # Replace tool list in request

    return handler(request)   # Execute the LLM call with (possibly modified) tools

In [7]:
# ============================================================
# CELL 5: Create the Agent With All Tools + Dynamic Filter
# ============================================================
# create_agent(tools=[web_search, sql_query]) gives the agent the FULL
# tool list as its default. The middleware then filters this list
# down per request based on user role.
#
# Think of it as:
#   create_agent → defines the MAXIMUM possible tool set
#   middleware   → enforces the ACTUAL tool set per request

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],   # Full tool set — filtered per request
    middleware=[dynamic_tool_call],  # Role-based access control
    context_schema=UserRole          # Declare context type
)

In [8]:
# ============================================================
# CELL 6: External User — SQL Is Invisible
# ============================================================
# context={"user_role": "external"}
#   → middleware sees "external", restricts tools to [web_search]
#   → LLM never sees sql_query in its tool list
#
# The question "How many artists are in the database?" requires
# internal DB access. Since sql_query is hidden, the agent either:
#   A) Uses web_search to find public information about Chinook
#   B) Tells the user it cannot access that information
#
# This is the correct security behaviour — the tool restriction
# works silently without any error message or special handling.

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}   # External role → sql_query hidden
)

print(response["messages"][-1].content)

There are 275 artists in the database.


---
## Summary

| Concept | Key takeaway |
|---|---|
| **Dynamic tools** | Filter the tool list per request based on user role — same agent, different capabilities |
| **`request.override(tools=[...])`** | Replace the tool list in the request before the LLM call |
| **Safe default** | Default `user_role="external"` — minimum permissions if role not provided |
| **LLM-level enforcement** | Model never sees restricted tools — prompt injection can't bypass it |
| **`pass` for permissive roles** | Internal users keep all tools — no explicit override needed |

### Extending to more roles

```python
@wrap_model_call
def dynamic_tool_call(request, handler):
    role = request.runtime.context.user_role
    if role == "admin":
        pass                                    # All tools
    elif role == "analyst":
        request = request.override(tools=[web_search, sql_query])
    elif role == "support":
        request = request.override(tools=[web_search, search_handbook])
    else:  # external
        request = request.override(tools=[web_search])
    return handler(request)
```

### Next steps
- **3.5 Email Agent** — combines dynamic tools, dynamic prompts, state, and HITL in one complete app